# Hands-On Agentic GraphRAG
# Notebook 2: Introduction to Knowledge Graphs

---
**Workshop series:** Agentic GraphRAG for Autonomous Incident Investigation  
**Prerequisites:** Notebook 1 — Vector RAG and Its Limitations  
**Runtime:** Google Colab — CPU only, no GPU required  

---

## Why We Need a Knowledge Graph

In Notebook 1, we saw that Vector RAG fails the moment a question requires crossing more than one relationship. The root cause is structural: flattening six JSON files into a list of sentences destroys every connection between entities. Once the data is flat, no amount of similarity search can recover those connections.

A **knowledge graph** solves this by keeping the structure intact. Instead of converting everything to sentences, we model the domain as a **directed property graph**:

- **Nodes** represent entities — services, teams, engineers, incidents, deployments
- **Edges** represent typed relationships — `OWNS`, `DEPENDS_ON`, `MEMBER_OF`, `TRIGGERED`, `AFFECTS`, `RESOLVED_BY`
- **Properties** on nodes and edges carry attributes — descriptions, timestamps, version numbers

With this structure in place, answering a multi-hop question is simply a matter of **traversing edges**, not searching for similar text.

---

### From Flat Sentences to a Graph

Compare how the same fact is represented in each approach:

| Approach | Representation | What is lost |
|---|---|---|
| Vector RAG | `"Identity Team owns Auth Service."` | Who the members are, what Auth Service depends on |
| Knowledge Graph | Edge: `Identity Team -[OWNS]-> Auth Service` | Nothing — the edge is one piece of a traversable structure |

The graph representation connects directly to every other fact about Identity Team (its members) and about Auth Service (its dependencies, its incidents, its deployments). A traversal can follow any of those edges in a single query.

---

### The Five Node Types in Our Graph

| Type | Examples | Key relationships |
|---|---|---|
| `service` | Auth Service, API Gateway | DEPENDS_ON, OWNS (incoming), AFFECTS (incoming) |
| `team` | Identity Team, Platform Team | OWNS (outgoing), MEMBER_OF (incoming), DEPLOYED_BY (incoming) |
| `member` | Alice, Omar, David | MEMBER_OF (outgoing), AUTHORED_BY (incoming), RESOLVED_BY (incoming) |
| `incident` | INC-001, INC-002, INC-003 | CAUSED_BY, AFFECTS, TRIGGERED (incoming), RESOLVED_BY |
| `deployment` | Auth Service v2.1, API Gateway v3.2.1 | DEPLOYED_TO, DEPLOYED_BY, AUTHORED_BY, TRIGGERED |

## Setup

In [ ]:
!pip install -q networkx matplotlib
print("Packages installed.")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

print("Libraries ready.")

## Building the Knowledge Graph

We construct the graph using NetworkX `DiGraph` — a directed graph where edges have a direction and carry a `rel` property for the relationship type.

The build follows five steps, each corresponding to a layer of operational knowledge:
1. Services
2. Teams and ownership
3. Team members
4. Service dependencies
5. Deployments and incidents with causal edges

Which graph model did we use here?

In [ ]:
G = nx.DiGraph()

# ── 1. Services ───────────────────────────────────────────────────────────────
G.add_node("Auth Service",
           type="service",
           description="Handles authentication and issues JWT tokens")
G.add_node("API Gateway",
           type="service",
           description="Entry point for all requests, routing and rate limiting")
G.add_node("Database Service",
           type="service",
           description="Stores credentials, session state, and audit logs")
G.add_node("Cache Service",
           type="service",
           description="Stores active sessions to reduce database load")

# ── 2. Teams ──────────────────────────────────────────────────────────────────
G.add_node("Identity Team",         type="team")
G.add_node("Platform Team",         type="team")
G.add_node("Data Engineering Team", type="team")
G.add_node("Infrastructure Team",   type="team")

G.add_edge("Identity Team",         "Auth Service",     rel="OWNS")
G.add_edge("Platform Team",         "API Gateway",      rel="OWNS")
G.add_edge("Data Engineering Team", "Database Service", rel="OWNS")
G.add_edge("Infrastructure Team",   "Cache Service",    rel="OWNS")

# ── 3. Team members ───────────────────────────────────────────────────────────
members = {
    "Alice":   "Identity Team",
    "Omar":    "Identity Team",
    "Lina":    "Identity Team",
    "David":   "Platform Team",
    "Sara":    "Platform Team",
    "John":    "Data Engineering Team",
    "Fatima":  "Data Engineering Team",
    "Michael": "Infrastructure Team",
    "Noor":    "Infrastructure Team",
}
for person, team in members.items():
    G.add_node(person, type="member", team=team)
    G.add_edge(person, team, rel="MEMBER_OF")

# ── 4. Service dependencies ───────────────────────────────────────────────────
G.add_edge("API Gateway",   "Auth Service",     rel="DEPENDS_ON")
G.add_edge("Auth Service",  "Database Service", rel="DEPENDS_ON")
G.add_edge("Auth Service",  "Cache Service",    rel="DEPENDS_ON")
G.add_edge("Cache Service", "Database Service", rel="DEPENDS_ON")

# ── 5. Deployments ────────────────────────────────────────────────────────────
deployments = [
    ("Auth Service v2.1",        "Auth Service",     "Alice",   "Identity Team"),
    ("Auth Error Handling",       "Auth Service",     "Lina",    "Identity Team"),
    ("DB Connection Pooling",     "Database Service", "John",    "Data Engineering Team"),
    ("DB Indexing Optimization",  "Database Service", "Fatima",  "Data Engineering Team"),
    ("Cache Service v1.4",        "Cache Service",    "Michael", "Infrastructure Team"),
    ("Cache TTL Tuning",          "Cache Service",    "Noor",    "Infrastructure Team"),
    ("API Gateway v3.2.1",        "API Gateway",      "David",   "Platform Team"),
    ("Adaptive Rate Limiting",    "API Gateway",      "Sara",    "Platform Team"),
]
for deploy, service, author, team in deployments:
    G.add_node(deploy, type="deployment")
    G.add_edge(deploy, service, rel="DEPLOYED_TO")
    G.add_edge(deploy, team,    rel="DEPLOYED_BY")
    G.add_edge(deploy, author,  rel="AUTHORED_BY")

# ── 6. Incidents ──────────────────────────────────────────────────────────────
G.add_node("INC-001", type="incident", summary="Login latency spike due to database overload")
G.add_node("INC-002", type="incident", summary="API Gateway dropping requests under high traffic")
G.add_node("INC-003", type="incident", summary="System-wide login failures after concurrent deployments")

# Root causes
G.add_edge("INC-001", "Database Service", rel="CAUSED_BY")
G.add_edge("INC-002", "API Gateway",      rel="CAUSED_BY")
G.add_edge("INC-003", "Auth Service",     rel="CAUSED_BY")

# Deployments that triggered INC-003
G.add_edge("Auth Service v2.1",  "INC-003", rel="TRIGGERED")
G.add_edge("API Gateway v3.2.1", "INC-003", rel="TRIGGERED")

# Affected services per incident
for svc in ["Database Service", "Auth Service"]:
    G.add_edge("INC-001", svc, rel="AFFECTS")
for svc in ["API Gateway", "Auth Service", "Database Service"]:
    G.add_edge("INC-002", svc, rel="AFFECTS")
for svc in ["Auth Service", "API Gateway", "Cache Service", "Database Service"]:
    G.add_edge("INC-003", svc, rel="AFFECTS")

# Resolved by
G.add_edge("INC-001", "Fatima", rel="RESOLVED_BY")
G.add_edge("INC-001", "Omar",   rel="RESOLVED_BY")
G.add_edge("INC-002", "Sara",   rel="RESOLVED_BY")
G.add_edge("INC-002", "Noor",   rel="RESOLVED_BY")
G.add_edge("INC-003", "Omar",   rel="RESOLVED_BY")
G.add_edge("INC-003", "Sara",   rel="RESOLVED_BY")

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print()

# Count by type
from collections import Counter
types = Counter(d.get("type") for _, d in G.nodes(data=True))
for t, count in sorted(types.items()):
    print(f"  {t:<15}  {count} nodes")

print()
print("Edge relationship types:")
for rel in sorted(set(d["rel"] for _, _, d in G.edges(data=True))):
    print(f"  {rel}")

## Schema Validation

Before doing anything with the graph, we check that every node has the fields it should have. This is the graph equivalent of checking for null values in a dataframe — catching structural errors early prevents silent failures in traversal queries later.

In [ ]:
REQUIRED_FIELDS = {
    "service":    ["type", "description"],
    "team":       ["type"],
    "member":     ["type", "team"],
    "incident":   ["type", "summary"],
    "deployment": ["type"],
}

errors = []
for node, data in G.nodes(data=True):
    node_type = data.get("type", "UNKNOWN")
    for field in REQUIRED_FIELDS.get(node_type, []):
        if field not in data:
            errors.append(f"  MISSING '{field}' on [{node_type}] node: {node}")

if errors:
    print("Schema FAILED:")
    for e in errors:
        print(e)
else:
    print("Schema PASSED — all nodes have required fields")

## Adding Temporal Properties

Timestamps are where a knowledge graph has a clear advantage over Vector RAG. In a flat corpus, dates are just substrings inside sentences with no structure. In a graph, timestamps are **properties on nodes** — which means we can filter, sort, and compare them programmatically.

We add `valid_from` and `valid_to` fields to every incident and deployment node. A `valid_to` of `None` means the deployment is still active (not rolled back).

This temporal layer enables queries like: *"what changed before this incident window?"*

In [ ]:
temporal = {
    "INC-001":                  {"valid_from": "2026-03-15T09:20", "valid_to": "2026-03-15T10:05"},
    "INC-002":                  {"valid_from": "2026-03-28T11:00", "valid_to": "2026-03-28T11:45"},
    "INC-003":                  {"valid_from": "2026-04-11T09:50", "valid_to": "2026-04-11T10:12"},
    "Auth Service v2.1":        {"valid_from": "2026-04-10T14:00", "valid_to": "2026-04-11T10:12"},
    "Auth Error Handling":      {"valid_from": "2026-04-10T14:05", "valid_to": None},
    "DB Connection Pooling":    {"valid_from": "2026-04-10T15:30", "valid_to": None},
    "DB Indexing Optimization": {"valid_from": "2026-04-10T15:45", "valid_to": None},
    "Cache Service v1.4":       {"valid_from": "2026-04-10T16:00", "valid_to": None},
    "Cache TTL Tuning":         {"valid_from": "2026-04-10T16:10", "valid_to": None},
    "API Gateway v3.2.1":       {"valid_from": "2026-04-11T08:30", "valid_to": "2026-04-11T10:12"},
    "Adaptive Rate Limiting":   {"valid_from": "2026-04-11T08:45", "valid_to": None},
}

for node, fields in temporal.items():
    G.nodes[node].update(fields)

print(f"{'Node':<30} {'valid_from':<20} {'valid_to':<20} {'Status'}")
print("-" * 80)
for node, data in G.nodes(data=True):
    if "valid_from" in data:
        status = "ACTIVE" if data["valid_to"] is None else "ROLLED BACK"
        print(f"  {node:<28} {data['valid_from']:<20} {str(data['valid_to']):<20} {status}")

## Visualizing the Graph

The visualization below renders the full graph with nodes colored by type. Use it to build an intuition for the structure before running traversal queries.

Node color legend:
- Blue — services
- Orange — teams
- Green — engineers (members)
- Red — incidents
- Purple — deployments

In [ ]:
TYPE_COLORS = {
    "service":    "#4C9BE8",
    "team":       "#E8914C",
    "member":     "#6DBE6D",
    "incident":   "#E86B6B",
    "deployment": "#B87FE8",
}

node_colors = [TYPE_COLORS.get(G.nodes[n].get("type"), "#cccccc") for n in G.nodes]

plt.figure(figsize=(18, 11))
plt.title("Incident Investigation Knowledge Graph", fontsize=14, fontweight="bold", pad=15)

pos = nx.spring_layout(G, seed=42, k=2.2)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1800, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold")
nx.draw_networkx_edges(
    G, pos, arrows=True, arrowsize=14,
    edge_color="#aaaaaa", width=0.9,
    connectionstyle="arc3,rad=0.08"
)
nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels={(u, v): d["rel"] for u, v, d in G.edges(data=True)},
    font_size=5.5, font_color="#444444"
)

legend = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()]
plt.legend(handles=legend, loc="upper left", fontsize=9)
plt.axis("off")
plt.tight_layout()
plt.savefig("knowledge_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph saved to knowledge_graph.png")

## Graph Traversal

This is the payoff section. We now run the exact same questions that Vector RAG could not answer, but this time using **graph traversal** instead of cosine similarity.

The traversal helpers below follow edges in the graph — no embeddings, no similarity scores, just structured path following.



In [ ]:
def out_neighbors(node, rel):
    """Return nodes reachable from `node` via outgoing edges with relationship `rel`."""
    return [t for _, t, d in G.out_edges(node, data=True) if d["rel"] == rel]


def in_neighbors(node, rel):
    """Return nodes that point TO `node` via edges with relationship `rel`."""
    return [s for s, _, d in G.in_edges(node, data=True) if d["rel"] == rel]


def show_path(*steps):
    """Print a traversal result as a readable chain."""
    print("  " + "  ->  ".join(str(s) for s in steps))


print("Traversal helpers ready.")

### Query Test

In [ ]:
question = "Which team owns the service that API Gateway depends on?"
print(f"Query: {question}")
print("-" * 72)

# Hop 1: API Gateway -> DEPENDS_ON -> services
for service in out_neighbors("API Gateway", "DEPENDS_ON"):
    # Hop 2: service <- OWNS <- team
    for team in in_neighbors(service, "OWNS"):

        print(f"\n  API Gateway")
        print(f"    -[DEPENDS_ON]->  {service}")
        print(f"    <-[OWNS]-        {team}")
        print(f"\n  Answer: {team}")

In [ ]:
question = "What was the full chain of events that led to INC-003?"
print(f"Query: {question}\n")

for deploy in in_neighbors("INC-003", "TRIGGERED"):
    author  = out_neighbors(deploy, "AUTHORED_BY")[0]
    service = out_neighbors("INC-003", "AFFECTS")[0]
    print(f"{author} → {deploy} → INC-003 → {service}")

In [ ]:
print("Query: What deployments happened before INC-003 started, and which teams were responsible?")
print("-" * 72)

inc003_start = datetime.fromisoformat(G.nodes["INC-003"]["valid_from"])
print(f"\n  INC-003 started: {inc003_start}")
print()

deployment_nodes = [
    n for n, d in G.nodes(data=True)
    if d.get("type") == "deployment" and d.get("valid_from")
]

before_incident = [
    n for n in deployment_nodes
    if datetime.fromisoformat(G.nodes[n]["valid_from"]) < inc003_start
]

before_incident.sort(key=lambda n: G.nodes[n]["valid_from"])

print(f"  {'Deployment':<30} {'Deployed at':<22} {'By team':<25} {'Author'}")
print("  " + "-" * 84)
for deploy in before_incident:
    ts     = G.nodes[deploy]["valid_from"]
    teams  = out_neighbors(deploy, "DEPLOYED_BY")
    authors = out_neighbors(deploy, "AUTHORED_BY")
    print(f"  {deploy:<30} {ts:<22} {str(teams[0] if teams else '?'):<25} {authors[0] if authors else '?'}")

In [ ]:
print("Query: Which deployments triggered INC-003 and which teams were responsible?")
print("-" * 72)

triggers = in_neighbors("INC-003", "TRIGGERED")
for deploy in triggers:
    ts      = G.nodes[deploy].get("valid_from", "?")
    teams   = out_neighbors(deploy, "DEPLOYED_BY")
    authors = out_neighbors(deploy, "AUTHORED_BY")
    target  = out_neighbors(deploy, "DEPLOYED_TO")
    print(f"\n  Deployment : {deploy}")
    print(f"    deployed to  : {target}")
    print(f"    deployed at  : {ts}")
    print(f"    team         : {teams}")
    print(f"    author       : {authors}")

In [ ]:
print("Query: What is the common root cause pattern across all three incidents, and which service is the single point of failure?")
print("-" * 72)

incidents = [n for n, d in G.nodes(data=True) if d.get("type") == "incident"]

from collections import Counter
cause_count  = Counter()
affect_count = Counter()

for inc in incidents:
    for svc in out_neighbors(inc, "CAUSED_BY"):
        cause_count[svc] += 1
    for svc in out_neighbors(inc, "AFFECTS"):
        affect_count[svc] += 1

print(f"\n  {'Service':<20} {'Root cause count':>18} {'Affected count':>16}")
print("  " + "-" * 56)
all_services = set(cause_count) | set(affect_count)
for svc in sorted(all_services, key=lambda s: -(cause_count[s] + affect_count[s])):
    print(f"  {svc:<20} {cause_count[svc]:>18} {affect_count[svc]:>16}")

spof = max(all_services, key=lambda s: cause_count[s] + affect_count[s])
print(f"\n  Single point of failure: {spof}")

### Share one traversal path that answers a multi-hop question

## Export the Graph

We save the graph as a GraphML file — a standard XML-based format that preserves node types, edge types, and all properties. This file will be loaded directly in Notebook 3 so we do not need to rebuild the graph from scratch.

In [ ]:
# GraphML does not support None values — replace with empty string before saving
for node, data in G.nodes(data=True):
    for key, value in list(data.items()):
        if value is None:
            G.nodes[node][key] = ""

nx.write_graphml(G, "incident_knowledge_graph.graphml")
print(f"Graph saved to incident_knowledge_graph.graphml")
print(f"  {G.number_of_nodes()} nodes  |  {G.number_of_edges()} edges")

## Summary

### What the Knowledge Graph Gives Us

| Query type | Vector RAG (Notebook 1) | Knowledge Graph (this notebook) |
|---|---|---|
| Multi-hop reasoning | Failed — no relationship traversal | Solved — follow edges across node types |
| Temporal filtering | Failed — timestamps are plain text | Solved — `valid_from` is a comparable datetime property |
| Causal chain reconstruction | Failed — no causal structure | Solved — `TRIGGERED`, `CAUSED_BY` edges make causality explicit |
| Global aggregation | Failed — top-k misses the full picture | Solved — iterate all nodes of a type |


In the next session, we focus on GraphRAG retrieval.

We move from manual traversal to a more structured approach that includes:

- Query understanding and decomposition
- Graph-guided retrieval over entities and relationships
- Community summaries for global questions

This is where we start solving global queries properly, using summaries and graph structure instead of raw path extraction.